In [11]:
import json
import torch
import librosa
import numpy as np
import pandas as pd
from torch.utils.data import Dataset, DataLoader
from transformers import AutoConfig, AudioFlamingo3ForConditionalGeneration

In [12]:
config = AutoConfig.from_pretrained("nvidia/music-flamingo-hf")
config

AudioFlamingo3Config {
  "architectures": [
    "AudioFlamingo3ForConditionalGeneration"
  ],
  "audio_config": {
    "activation_dropout": 0.0,
    "activation_function": "gelu",
    "attention_dropout": 0.0,
    "dropout": 0.0,
    "hidden_size": 1280,
    "initializer_range": 0.02,
    "intermediate_size": 5120,
    "layerdrop": 0.0,
    "max_source_positions": 1500,
    "model_type": "audioflamingo3_encoder",
    "num_attention_heads": 20,
    "num_hidden_layers": 32,
    "num_mel_bins": 128,
    "scale_embedding": false
  },
  "audio_token_id": 151669,
  "dtype": "bfloat16",
  "model_type": "audioflamingo3",
  "projector_bias": true,
  "projector_hidden_act": "gelu",
  "text_config": {
    "attention_dropout": 0.0,
    "bos_token_id": 151670,
    "eos_token_id": 151645,
    "hidden_act": "silu",
    "hidden_size": 3584,
    "initializer_range": 0.02,
    "intermediate_size": 18944,
    "layer_types": [
      "full_attention",
      "full_attention",
      "full_attention",
      "

In [39]:

config.audio_config.hidden_size = 32
config.audio_config.intermediate_size = 64
config.audio_config.num_attention_heads = 2
config.audio_config.num_hidden_layers = 2

config.text_config.num_hidden_layers = 2
config.text_config.layer_types = ["full_attention", "full_attention"]
config.text_config.max_window_layers = 2
config.text_config.num_attention_heads = 4
config.text_config.num_key_value_heads = 1
config.text_config.hidden_size = 32
config.text_config.intermediate_size = 64

In [40]:
model = AudioFlamingo3ForConditionalGeneration(config)
# model.train()

print(f"Parameters: {sum(p.numel() for p in model.parameters()) / 1e6:.1f}M")

Parameters: 9.8M


In [38]:
df = pd.read_csv('test.csv')

with open("conversations.jsonl", "w", encoding="utf-8") as f:
    for i, row in df.iterrows():
        conversation = [
            {
                "role": "user",
                "content": [
                    {
                        "type": "text",
                        "text": """Please rate the intensity with wich you felt each of the following feelings in this music excerpt, on a scale ranging from 1 (not at all) to 5 (very much).
Just give the ratings, without justifying.
- Wonder (Filled with wonder, Dazzled, Allured, Moved)
- Transcendence (Fascinated, Overwhelmed, Feelings of transcendence and spirituality)
- Nostalgia (Nostalgic, Dreamy, Sentimental, Melancholic)
- Tenderness (Tender, Affectionate, In love, Mellowed)
- Peacefulness (Serene, Calm, Soothed, Relaxed)
- Joy (Joyful, Amused, Animated, Bouncy)
- Sadness (Sad, Sorrowful)
- Power (Strong, Triumphant, Energetic, Fiery)
- Tension (Tense, Agitated, Nervous, Irritated)
                        """
                    },
                    {
                        "type": "audio",
                        "path": row["Path"],
                    },
                ],
            },
            {
                "role": "assistant",
                "content": [
                    {
                        "type": "text",
                        "text": (
                            f"- Wonder: {int(row['Wonder'])}\n"
                            f"- Transcendence: {int(row['Transcendence'])}\n"
                            f"- Nostalgia: {int(row['Nostalgia'])}\n"
                            f"- Tenderness: {int(row['Tenderness'])}\n"
                            f"- Peacefulness: {int(row['Peacefulness'])}\n"
                            f"- Joy: {int(row['Joy'])}\n"
                            f"- Sadness: {int(row['Sadness'])}\n"
                            f"- Power: {int(row['Power'])}\n"
                            f"- Tension: {int(row['Tension'])}"
                        ),
                    }
                ],
            },
        ]

        json_line = json.dumps({"conversation": conversation}, ensure_ascii=False)
        f.write(json_line + "\n")

In [41]:
class EmotionMusicDataset(Dataset):
    def __init__(self, json_path, sample_rate=16000):
        with open(json_path, "r") as f:
            self.data = [json.loads(line) for line in f]
        self.sample_rate = sample_rate

    def __len__(self):
        return len(self.data)

    def __getitem__(self, idx):
        conversation = self.data[idx]["conversation"]

        resolved = []
        for turn in conversation:
            new_turn = {"role": turn["role"], "content": []}
            for item in turn["content"]:
                if item["type"] == "audio":

                    audio, _ = librosa.load(item["path"], sr=self.sample_rate, mono=True)
                    new_turn["content"].append({
                        "type": "audio",
                        "array": audio,
                        "sampling_rate": self.sample_rate
                    })
                else:
                    new_turn["content"].append(item)
            resolved.append(new_turn)

        return resolved

In [42]:
class CollateFunction:
    def __init__(self, processor):
        self.processor = processor

    def __call__(self, batch):
        inputs = self.processor.apply_chat_template(
            batch,
            tokenize=True,
            add_generation_prompt=False,
            return_dict=True,
            return_tensors="pt",
            output_labels=True,
            padding=True,
        )
        return inputs

In [34]:
def move_to_device(obj, device):
    if isinstance(obj, torch.Tensor):
        return obj.to(device)
    elif isinstance(obj, dict):
        return {k: move_to_device(v, device) for k, v in obj.items()}
    elif isinstance(obj, list):
        return [move_to_device(v, device) for v in obj]
    return obj

In [18]:
from transformers import AudioFlamingo3ForConditionalGeneration, AutoProcessor
from torch.optim import AdamW
from torch.optim.lr_scheduler import CosineAnnealingLR

In [45]:
MODEL_ID   = "nvidia/music-flamingo-hf"
JSON_PATH  = "conversations.jsonl"
OUTPUT_DIR = "./music-flamingo-emotion"
EPOCHS     = 5
BATCH_SIZE = 2
LR         = 2e-5
DEVICE     = "cuda" if torch.cuda.is_available() else "cpu"
print(DEVICE)

cuda


In [20]:
processor = AutoProcessor.from_pretrained(MODEL_ID)
# model = AudioFlamingo3ForConditionalGeneration.from_pretrained(
#     MODEL_ID,
#     torch_dtype=torch.bfloat16,
#     device_map="auto",
# )
model = AudioFlamingo3ForConditionalGeneration(config)

In [30]:
dataset    = EmotionMusicDataset(JSON_PATH)
collate_fn = CollateFunction(processor)
dataloader = DataLoader(
    dataset,
    batch_size=BATCH_SIZE,
    shuffle=True,
    collate_fn=collate_fn,
    num_workers=0,
)

In [32]:
trainable_params = [p for p in model.parameters() if p.requires_grad]
optimizer  = AdamW(trainable_params, lr=LR, weight_decay=0.01)
scheduler  = CosineAnnealingLR(optimizer, T_max=EPOCHS * len(dataloader))

In [46]:
model.train()
for epoch in range(EPOCHS):
    total_loss = 0
    for step, batch in enumerate(dataloader):
        # batch = {k: v.to(DEVICE) if isinstance(v, torch.Tensor) else v
        #          for k, v in batch.items()}
        batch = move_to_device(batch, DEVICE)
        # for k, v in batch.items():
        #     if isinstance(v, torch.Tensor):
        #         print(k, v.device)
        
        # print(f"batch : {step}")
        outputs   = model(**batch)
        
        # print(f"Forward : {step}")
        loss      = outputs.loss

        loss.backward()
        torch.nn.utils.clip_grad_norm_(trainable_params, max_norm=1.0)
        optimizer.step()
        scheduler.step()
        optimizer.zero_grad()

        total_loss += loss.item()
        if step % 10 == 0:
            print(f"Epoch {epoch+1} | Step {step} | Loss: {loss.item():.4f}")

    avg = total_loss / len(dataloader)
    print(f"Epoch {epoch+1} complete — avg loss: {avg:.4f}")

    # Save checkpoint after each epoch
    ckpt_dir = f"{OUTPUT_DIR}/epoch-{epoch+1}"
    model.save_pretrained(ckpt_dir)
    processor.save_pretrained(ckpt_dir)
    print(f"Checkpoint saved : {ckpt_dir}")

Epoch 1 | Step 0 | Loss: 11.9408
Epoch 1 | Step 10 | Loss: 11.9410
Epoch 1 complete — avg loss: 11.9403


Writing model shards: 100%|██████████| 1/1 [00:00<00:00, 26.53it/s]


Checkpoint saved : ./music-flamingo-emotion/epoch-1
Epoch 2 | Step 0 | Loss: 11.9402
Epoch 2 | Step 10 | Loss: 11.9408
Epoch 2 complete — avg loss: 11.9403


Writing model shards: 100%|██████████| 1/1 [00:00<00:00, 28.35it/s]


Checkpoint saved : ./music-flamingo-emotion/epoch-2
Epoch 3 | Step 0 | Loss: 11.9407
Epoch 3 | Step 10 | Loss: 11.9382
Epoch 3 complete — avg loss: 11.9403


...

Checkpoint saved : ./music-flamingo-emotion/epoch-3
Epoch 4 | Step 0 | Loss: 11.9396
Epoch 4 | Step 10 | Loss: 11.9401
Epoch 4 complete — avg loss: 11.9403


Writing model shards: 100%|██████████| 1/1 [00:00<00:00, 36.39it/s]


Checkpoint saved : ./music-flamingo-emotion/epoch-4
Epoch 5 | Step 0 | Loss: 11.9398
Epoch 5 | Step 10 | Loss: 11.9424
Epoch 5 complete — avg loss: 11.9403


Writing model shards: 100%|██████████| 1/1 [00:00<00:00, 38.97it/s]


Checkpoint saved : ./music-flamingo-emotion/epoch-5


In [ ]:
model.save_pretrained(OUTPUT_DIR)
processor.save_pretrained(OUTPUT_DIR)
print(f"Final model saved → {OUTPUT_DIR}")